# Notebook 6: Numerical instabilities in particle-in-cell codes
## using the ZPIC particle-in-cell code

## Submission
The assessed parts of this computing laboratory are due Friday 14th November at 11:59PM.

## Classwork

In this Worksheet 6 we will break our code to understand its limits. We will examine the stability of the pusher. We will study the effect of the interpolation scheme. Finally, we will look at numerical heating (the grid instability) arising from the aliasing of forces and how to minimize it.

In [ ]:
# Libraries needed for this worksheet
import numpy as np
import matplotlib.pyplot as plt
# Add zpic library to path
import sys
sys.path.append("../../lib")


import es1d
import lshelper as ls6


## Part 1: Instability due to the pusher algorithm

The pusher for the ES1D model is very simple, being a straightforward (not having a magnetic field involved) split step (leapfrog) ODE integrator algorithm over a time step $\Delta t$. The leapfrog algorithm is center differenced in time for both velocity and position with them being scattered with respect to each other. 

<div style="background-color: #013220" > 
<span style="color:white;">

**Exercise 1: (Revision) write down the leapfrog scheme for the particle pusher in terms of $v^{n\pm1/2}$, $x^{n+1}$, $x^n$, $E^n$ etc.**

</span>


Now we will do simulations with different timesteps to study the stability of the particle pusher. First we define a typical cold plasma wave simulation similar to Labscript 5

In [ ]:
# Simulation parameters
nx = 128            # Number of grid cells
box = 2.0 * np.pi   # Length of the box
dt = 0.1            # Time step
ppc = 4             # Particles per cell
tmax = 10 * np.pi   # Maximum time of the simulation
vth = 0.0
q_e = -1.0           # Electron charge

# Custom density profile - can be some function of x
# here it is just a constant density n0
def profile(x):
    n0 = 1.0
    return n0 

density = es1d.Density( type = "custom", custom = profile )

# Initialize the species
electrons = es1d.Species( "electrons", q_e, ppc, density = density, vth=vth)

# Initialize the simulation
sim = es1d.Simulation( nx, box, dt,species = electrons)

# This helper function obtains the particle positions in the full simulation box xi
# Note that the particles are stored in cell number "ix" with relative position offset 
# "x" in the cell, so the actual position is (ix + x) * dx which this function returns
xi = ls6.xpos(electrons)

# Initialize the particle velocities to obtain a wave with density perturbation
#  magnitude delta_n0  =0.01 and a sinusoidal perturbation


Now we wish to track the total kinetic energy of the particles and field energy integrated over the simulation box to be able to track the energy conservation as a function of time. To do this we will build a field energy diagnostic

$$
U^n = \frac{1}{2} \sum_j \left\{E^n_j\right\}^2 \Delta x
$$

and a particle total kinetic energy diagnostic

$$
K^n = \frac{1}{2} \sum_i w_iv_i^{n+1/2}v_i^{n-1/2} \Delta x
$$

where $w_i$ is the weight of the particle. Note that the velocity combination is like it is above so that we have the kinetic energy time centered at $n$.

<div style="background-color: #013220" > 
<span style="color:white;">

**Exercise 2: Write a diagnostic ```KE_time(sim,species,tmax)``` that iterates to ```tmax``` similar to ```run_chg_E_time(sim, tmax)``` in Labscript 5 but generates a time series of the total kinetic energy and the total field energy starting from the function header below.**

</span>
</div>

**Note that the time centering and correct scaling matters! If you get it right, you should obtain something like the figure below. Note that the total energy is constant.**
<div align="center">
  <img src="ls6_figures/energy_conserve_cold_plasma.png" alt="new_test1" width="600px">

*Figure 1: The time series of kinetic and electrostatic energy in the plasma wave*

</div>


In [ ]:

def KE_time(sim,species,tmax):
        # code here

In [ ]:
# This function call generates the energy time series
KE_t,E2_t,time = KE_time(sim,electrons,tmax)
plt.figure(figsize=(10, 4))
plt.plot(time,KE_t,'b',time,E2_t,'r',time,KE_t+E2_t,'k')
plt.xlabel('$\omega_pt$')
plt.legend(('Kinetic energy','Field energy','Total energy'),loc='upper right',framealpha=1)

<div style="background-color: #013220" > 
<span style="color:white;">

**Exercise 3: Now loop over a series of simulations with fixed time but varying the timestep size from $\Delta t=0.1$ to  to examine the stability condition. You should observe a very pronounced increase in the total energy at the end of the simulation once you pass a threshold, similar to the figure below.**

The figure shows the total energy in the system minus the initial total energy, i.e. the error in the energy conservation. 

Now vary the density n0 (in the species initialization as in the box above). How does the threshold for the instability change? Can you relate this to the stability conditions we learned at the beginning of the course? Report the results and conclusions about the stability in the box below

</span>
</div>
<span style="color:red;">

**WARNING! The range of $\Delta t$ below is carefully chosen. As you change $n_0$ you should think about the range of $\Delta t$ to iterate over. Unfortunately, when the code goes too unstable it throws a memory error that at least for me kills the python kernel so you want to avoid that.**

</span>


<div align="center">
  <img src="ls6_figures/energy_dt.png" alt="new_test2" width="600px">

*Figure 2: The error in the total energy as a function of $\Delta t$ for fixed simulation of time $t_{max}=20$.*

</div>

In [ ]:
vth = 0.0001
tmax = 20.0
# Initialize the species
electrons = es1d.Species( "electrons", q_e, ppc, vth=vth)

dts = np.linspace(0.1,2.5,20)

for dt in dts:
    # Initialize the simulation
    sim = es1d.Simulation( nx, box, dt,species = electrons)

    # code here

In [ ]:
# Plot the total energy minus initial energy
plt.semilogy(dts,np.abs(KE_dt+E2_dt - (KE_dt+E2_dt)[0]),'k')
plt.xlabel('$\Delta t$')
plt.ylabel('Error in total energy')

## Part 2: The effect of the particle weighting scheme

As we saw in lectures, the interpolation scheme used to map the particle charge to the grid and the gridded fields to the force on a particle can strongly affect the numerical physics. Here, we look at the effect of the weighting schemes on the charge and electric field on the grid and then examine its filtering properties. First, we look at the shape of a single particle.

In [ ]:
# Simulation parameters
nx = 16             # Number of grid cells - low resolution!
box = 1.0           # Length of the box
dt = 0.1            # Time step
ppc = 1             # Particles per cell
vth = 0.0           # thermal velocity
q_e = -1.0          # Electron charge

# Initialize the species
electrons = es1d.Species( "electrons", q_e, ppc, vth=vth)

# Initialize the simulation
# The additional argument spline_order is equal to 0,1,2 and
# corresponds to:
# 0: Nearest neighbor weighting (0th order B-sline)
# 1: Linear particle weighting (1st order B-sline)
# 2: Quadratic particle weighting (2nd order B-sline)
sim = es1d.Simulation( nx, box, dt,species = electrons,spline_order=1)

# Set the initial particle positions all to be the same
# i.e. apart from the magnitude this is identical to 
# a single particle
electrons.particles['ix'] = 7
electrons.particles['x'] = 0.0
sim.iter()

The box below prints out the particle shape on the grid, i.e. the charge density of a single particle, and its resulting electric field profile, along with the spatial limits of the simulation.

In [ ]:
x = np.linspace(0,sim.box,sim.nx)
mylims = [min(sim.field.E),max(np.abs(sim.charge.rho/nx))]
plt.plot(x,sim.field.E,'ro',x,np.abs(sim.charge.rho/nx),'bo',x,sim.field.E,'m-',x,np.abs(sim.charge.rho/nx),'c-',[0.5,0.5],mylims,'k--',[0,0],mylims,'k-',[sim.box,sim.box],mylims,'k-')
plt.axis='tight'
plt.legend(('charge density','electric field'),loc = 'upper right')
plt.xlabel('$x\omega_p/c$')

<div style="background-color: #013220" > 
<span style="color:white;">

**Exercise 4: Look at the density and electric field for the three different weighting schemes. Move the particle backwards and forwards through the cell to see how the charge profile on the grid varies with the particle position.**

</span>
</div>

Next, we look at the smoothness of the particle density with the different shape factors for low resolution waves to see the smoothing effect.

In [ ]:
# Simulation parameters
nx = 32            # Number of grid cells
box = 2.0 * np.pi   # Length of the box
dt = 0.1            # Time step
ppc = 2           # Particles per cell
tmax = 0.5*np.pi # Maximum time of the simulation
vth = 0.0
q_e = -1.0           # Electron charge

# Initialize the species
electrons = es1d.Species( "electrons", q_e, ppc, vth=vth)


<div style="background-color: #013220" > 
<span style="color:white;">

**Exercise 5: Iterate over the spline order for sinusoidal waves of different amplitudes from 0.01 to 0.1. Try different numbers of particles per cell. Generate 3 subplots in a figure showing the particle charge density as a function of x for the 3 different spline schemes.**

</span>
</div>

In [ ]:
fig = plt.figure(figsize=[12,4])
subfigs = fig.subfigures(1, 3)

for so in range(3):
    # Initialize the simulation
    sim = es1d.Simulation( nx, box, dt,species = electrons, spline_order=so)

    # code here


## Part 3: Grid Heating instability and interpolation schemes

 In the lectures we saw that in theory, the particle-grid interactions lead to fictitious forces that  lead to particle self-heating. We learned that this can be limited by higher order weighting schemes, or when the Debye length is resolved and emperically how the number of particles per cell can reduce this. In Ueda et al, CPC 1993, the authors carried out an emperical study of the self-heating of a plasma, as a function of various parameters, for example with the scalings shown in the figure below.
 
 <div align="center">
  <img src="ls6_figures/Ueda_fig.png" alt="new_test2" width="600px">

*Figure 3: Experimental results of the heating rate $\langle r\rangle$ vs (a) $N_p/N_x$ and (b) $\lambda_D/\Delta x$. (From Ueda et al., CPC 1993.*

</div>

 
 We will now show that the fictitious forces lead to particle heating in your system in practice.

 <div style="background-color: #013220" > 
<span style="color:white;">

**Exercise 6: Investigate the effect of different parameters on numerical heating. How does the following affect the self-heating rate?**
* $\lambda_D/\Delta x$ (by varying the thermal velocity $v_{\th}$)
* Number of particles per cell?
* The spline interpolation method?

Start from the following input parameters (that are based on Ueda CPC) and allow the simulation to run for ```tmax = 300```. Now take your energy vs time diagnostic ```KE_t,E2_t,time = KE_time(sim,electrons,tmax)``` and plot the total energy vs time. Fit a straight line to the slope of the logarithm of the total energy $\log(K+U)$ to obtain the heating rate for those parameters. By varying the parameters generate graphs similar to the figure above, and in addition for the spline interpolation method. Write a short report on your findings in the box below.

</span>
</div>

In [ ]:
nx = 256
box = 512 
dt = 0.02
tmax = 300

ppc = 1
vth = 1.0 

electrons = es1d.Species( "electrons", -1.0, ppc, vth = vth )

print("\lambda_D/\Delta x = %.2f " % (vth/sim.dx))

# Initialize the simulation without diagnostics
sim = es1d.Simulation( nx, box, dt, species = electrons, spline_order= 1)





You could also look at the thermal fluctations as a function of time using your ```run_chg_E_time(sim, tmax)``` diagnostic.

In [ ]:
# This function will run the simulation and create data arrays for charge and field
def run_chg_E_time(sim, tmax):
    charge_data_time = (sim.charge.rho)
    E_data_time = (sim.field.E)
    while ((sim.t<tmax) and (sum(sim.field.E**2)*sim.dx<1)):
        sim.iter()
        charge_data_time = np.vstack((charge_data_time,sim.charge.rho))
        E_data_time = np.vstack((E_data_time,sim.field.E))
    return np.rot90(charge_data_time), np.rot90(E_data_time)

charge_data_time, E_data_time = run_chg_E_time(sim, tmax)
ls6.plot_field_and_charge(sim, charge_data_time, E_data_time, sim.t)
   


## Write a short report in the box below on your findings, including supporting figures.